In [ ]:
import os
from pathlib import Path
import pydicom
import matplotlib.pyplot as plt
import textwrap

# 1. 탐색할 최상위 디렉토리 경로 설정
#root_dir = Path("C:\\Users\\csm02\\Desktop\\edward\\bmd\\etc\\dataset\\kaggle_archive\\physionet.org\\files\\vindr-spinexr\\1.0.0\\train_images")
#root_dir = Path("C:\\Users\\csm02\\Desktop\\edward\\bmd\\etc\\dataset\\dataset-dcm\\lateral-pin")
#root_dir = Path("C:\\Users\\csm02\\Desktop\\edward\\bmd\\etc\\dataset\\dataset-dcm\\ap-pin")
root_dir = Path("C:\\Users\\csm02\\Desktop\\edward\\bmd\\etc\\dataset\\dataset-dcm\\lateral-pin\\")
    
# 2. 하위 폴더를 모두 뒤져서 .dcm 파일 탐색 (.dcm, .DCM 모두 포함)
dcm_files = sorted(list(set(root_dir.rglob("*.dcm")) | set(root_dir.rglob("*.dicom"))))

if not dcm_files:
    print(f"'{root_dir}' 하위 경로에서 .dcm 파일을 찾지 못했습니다.")
else:
    total_files = len(dcm_files)
    print(f"총 {total_files}개의 DICOM 파일을 찾았습니다.")
    
    # --- [수정 포인트] 배치(Batch) 설정 ---
    chunk_size = 100  # 한 번에 보여줄 파일 개수
    images_per_row = 4  # 한 줄에 보여줄 이미지 개수
    
    # 전체 데이터를 100개씩 나누어 처리
    for start_idx in range(0, total_files, chunk_size):
        end_idx = min(start_idx + chunk_size, total_files)
        chunk_files = dcm_files[start_idx:end_idx]
        chunk_len = len(chunk_files)
        
        print(f"\n>>> 시각화 중: {start_idx + 1}번째부터 {end_idx}번째 파일 (총 {chunk_len}개)")
        
        # 현재 배치에 필요한 총 행(row) 수 계산
        num_rows = (chunk_len + images_per_row - 1) // images_per_row
        
        # 현재 배치의 Figure 생성
        fig, axes = plt.subplots(num_rows, images_per_row, figsize=(20, 5 * num_rows))
        
        # axes 배열 평탄화 처리
        if chunk_len == 1:
            axes = [axes]
        else:
            axes = axes.flatten()
            
        # 현재 배치 파일 순회하며 그리기
        for i, file_path in enumerate(chunk_files):
            ax = axes[i]
            wrapped_title = "\n".join(textwrap.wrap(file_path.name, width=20))
            
            try:
                ds = pydicom.dcmread(file_path)
                img = ds.pixel_array
                ax.imshow(img, cmap='gray')
                ax.set_title(wrapped_title, fontsize=10, pad=8)
                
            except Exception as e:
                ax.text(0.5, 0.5, f"Error\n{file_path.name}", 
                        ha='center', va='center', color='red')
                ax.set_title(wrapped_title, fontsize=10, pad=8)
            
            ax.axis('off')
            
        # 남는 빈 칸(Plot) 축 숨기기
        for j in range(i + 1, len(axes)):
            axes[j].axis('off')
            
        # 출력 및 메모리 정리
        plt.tight_layout()
        plt.show()
        
        # 메모리 누수 방지를 위해 현재 그림 완전히 닫기
        plt.close(fig)

총 21개의 DICOM 파일을 찾았습니다.

>>> 시각화 중: 1번째부터 21번째 파일 (총 21개)
